# Transformer (GPT-style) in PyTorch — Theory + Practice
This notebook builds a GPT-style Transformer using PyTorch.

Each step includes:
- Concept explanation
- Mathematical formulation
- Code implementation


## Step 0: Imports
We use PyTorch for automatic differentiation and efficient training.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)

## Step 1: Tokenization
Convert text into tokens.

No math here — just mapping words → integers.

In [2]:
sentences = [
    "i love ai",
    "i love machine learning",
    "ai is powerful",
    "machine learning is fun",
    "i enjoy learning"
]

vocab = {}
for s in sentences:
    for w in s.split():
        if w not in vocab:
            vocab[w] = len(vocab)

inv_vocab = {i:w for w,i in vocab.items()}

def tokenize(s):
    return [vocab[w] for w in s.split()]

data = [tokenize(s) for s in sentences]
vocab_size = len(vocab)
print(vocab)

{'i': 0, 'love': 1, 'ai': 2, 'machine': 3, 'learning': 4, 'is': 5, 'powerful': 6, 'fun': 7, 'enjoy': 8}


## Step 2: Embeddings

**Formula:**
$$x = Embedding(token)$$

Maps token IDs to dense vectors.

**Significance:**
- Captures semantic meaning
- Learnable representation space

In [3]:
d_model = 32
embedding = nn.Embedding(vocab_size, d_model)
embedding

Embedding(9, 32)

## Step 3: Positional Encoding

**Formula:**
$$PE(pos, 2i) = sin(pos / 10000^{2i/d_model})$$
$$PE(pos, 2i+1) = cos(pos / 10000^{2i/d_model})$$

**Significance:**
- Injects order information
- Enables sequence understanding

In [4]:
def positional_encoding(seq_len, d_model):
    PE = torch.zeros(seq_len, d_model)
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i] = torch.sin(torch.tensor(pos / (10000 ** (i / d_model))))
            if i+1 < d_model:
                PE[pos, i+1] = torch.cos(torch.tensor(pos / (10000 ** (i / d_model))))
    return PE

positional_encoding(16, d_model)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  5.3317e-01,  8.4601e-01,  3.1098e-01,
          9.5042e-01,  1.7689e-01,  9.8423e-01,  9.9833e-02,  9.9500e-01,
          5.6204e-02,  9.9842e-01,  3.1618e-02,  9.9950e-01,  1.7782e-02,
          9.9984e-01,  9.9998e-03,  9.9995e-01,  5.6234e-03,  9.9998e-01,
          3.1623e-03,  9.9999e-01,  1.7783e-03,  1.0000e+00,  1.0000e-03,
          1.0000e+00,  5.6234e-04,  1.0000e+00,  3.1623e-04,  1.0000e+00,
          1.7783e-04,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  9.02

## Step 4: Scaled Dot-Product Attention

**Formula:**
$$Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$$

**Significance:**
- Learns relationships between tokens
- Core of transformer architecture

In [5]:
class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=-1)
        return weights @ V

## Step 5: GPT Block

Structure:
Masked Attention → Add & Norm → FFN → Add & Norm

**Significance:**
- Residual connections stabilize training
- FFN adds non-linearity

In [8]:
class GPTBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = Attention(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.ReLU(),
            nn.Linear(d_model*4, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        attn_out = self.attn(x, mask)
        x = self.norm1(x + attn_out)

        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)

        return x

## Step 6: Full GPT Model

Stack multiple blocks.

In [9]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([GPTBlock(d_model) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        mask = torch.tril(torch.ones(seq_len, seq_len))

        x = self.embed(x)
        x = x + positional_encoding(seq_len, d_model)

        for layer in self.layers:
            x = layer(x, mask)

        return self.fc(x)

## Step 7: Training

**Loss:** Cross Entropy
$$L = -log(p_{correct})$$

**Significance:**
- Measures prediction error
- Drives learning via gradients

In [10]:
model = GPT(vocab_size, d_model)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

def create_dataset(data):
    X, Y = [], []
    for tokens in data:
        for i in range(1, len(tokens)):
            X.append(tokens[:i])
            Y.append(tokens[i])
    return X, Y

X_train, Y_train = create_dataset(data)

for epoch in range(50):
    total_loss = 0
    for x,y in zip(X_train, Y_train):
        x = torch.tensor(x).unsqueeze(0)
        y = torch.tensor([y])

        logits = model(x)
        loss = loss_fn(logits[:, -1, :], y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 31.5674
Epoch 10, Loss: 10.9716
Epoch 20, Loss: 3.9250
Epoch 30, Loss: 3.7368
Epoch 40, Loss: 3.6752


## Step 8: Text Generation
Generate next tokens iteratively.

In [18]:
def generate(start, max_len=5):
    tokens = tokenize(start)

    for _ in range(max_len):
        x = torch.tensor(tokens).unsqueeze(0)
        logits = model(x)
        next_token = torch.argmax(logits[0,-1]).item()
        tokens.append(next_token)

    return " ".join([inv_vocab[t] for t in tokens])

def guard_rails(logits):
    pass

print(generate("powerful", 4))

powerful learning is fun is
